In [1]:
# Mount Google Drive

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
#extract data ke local colabs
!tar -xzf "/content/drive/MyDrive/Colab Notebooks/Data/liputan6_data.tar.gz" -C "/content/sample_data"


In [10]:
# Load Dataset

import json
import os
import random

train_path = "/content/sample_data/liputan6_data/canonical/train"
dev_path = "/content/sample_data/liputan6_data/canonical/dev"
test_path = "/content/sample_data/liputan6_data/canonical/test"

#fungsi load urutan
def load_all_json(folder):
    data = []

    files = sorted(os.listdir(folder))[:10]

    for filename in files:
        if filename.endswith(".json"):
            file_path = os.path.join(folder, filename)

            with open(file_path, 'r') as f:
                item = json.load(f)
                data.append(item)

    return data

# fungsi load random
def load_random_json(folder, n=10):
    data = []

    files = [f for f in os.listdir(folder) if f.endswith(".json")]

    sampled_files = random.sample(files, n)  # random tanpa duplikat

    for filename in sampled_files:
        file_path = os.path.join(folder, filename)

        with open(file_path, 'r') as f:
            item = json.load(f)
            data.append(item)

    return data

# dimatikan karena akan menggunakan random, jika ada keperluan silahkan buka
# jika dihidupkan, jangan lupa matikan random dibawah

#train_data = load_all_json(train_path)
#dev_data = load_all_json(dev_path)
#test_data = load_all_json(test_path)

train_data = load_random_json(train_path)
dev_data = load_random_json(dev_path)
test_data = load_random_json(test_path)

print("Jumlah data train_data:", len(train_data))
print("Contoh data train_data:", train_data[0])
print("--"*50)
print("Jumlah data dev_data:", len(dev_data))
print("Contoh data dev_data:", dev_data[0])
print("--"*50)
print("Jumlah data test_data:", len(test_data))
print("Contoh data test_data:", test_data[0])

Jumlah data train_data: 5000
Contoh data train_data: {'id': 67433, 'url': 'https://www.liputan6.com/news/read/67433/korban-banjir-malang-kesulitan-menjual-hasil-kebun', 'clean_article': [['Liputan6', '.', 'com', ',', 'Malang', ':', 'Ratusan', 'warga', 'di', 'sejumlah', 'dusun', 'di', 'Desa', 'Pujiharjo', ',', 'Kecamatan', 'Tirtoyudo', ',', 'Kabupaten', 'Malang', ',', 'Jawa', 'Timur', ',', 'kesulitan', 'menjual', 'pisang', 'yang', 'menjadi', 'mata', 'pencaharian', 'sehari-hari', 'setelah', 'terisolasi', 'delapan', 'hari', 'akibat', 'banjir', '.'], ['Ini', 'dikarenakan', 'belum', 'ada', 'kendaraan', 'umum', 'yang', 'masuk', 'ke', 'wilayahnya', 'sejak', '34', 'titik', 'tanah', 'longsor', 'menutupi', 'jalan', 'sepanjang', '15', 'kilometer', '.'], ['"', 'Selama', 'sepekan', ',', 'penduduk', 'di', 'tiga', 'RT', 'tidak', 'bisa', 'keluar', ',', '"', 'ujar', 'Dai', 'Haris', ',', 'seorang', 'warga', ',', 'baru-baru', 'ini', '.'], ['Penduduk', 'yang', 'kehilangan', 'pekerjaan', 'tak', 'bisa', 'be

In [11]:
# Convert Dataset Dan Ambil clean_artikel , clean_summary, Dan Extractive saja

def convert_to_text(data):
    new_data = []

    for item in data:
        # Gabungkan artikel
        article = " ".join(
            [" ".join(sentence) for sentence in item["clean_article"]]
        )

        # Gabungkan summary
        summary = " ".join(
            [" ".join(sentence) for sentence in item["clean_summary"]]
        )

        # 🔥 ambil  (index)
        id = item["id"]

        # 🔥 ambil extractive summary (index)
        extractive = item["extractive_summary"]

        new_data.append({
            "id": id,
            "clean_article": article,
            "clean_summary": summary,
            "extractive_summary": extractive,
        })

    return new_data

train_data_convert = convert_to_text(train_data)
dev_data_convert = convert_to_text(dev_data)
test_data_convert = convert_to_text(test_data)

In [8]:
train_data_convert

[{'id': 159658,
  'clean_article': 'Liputan6 . com , Ambon : Nataniel Elake , Pembantu Rektor ( Purek ) II Sekolah Tinggi Agama Kristen Protestan Negeri Ambon , Maluku , tak menyangka dirinya harus berurusan dengan polisi . Belum lama ini dia mengancam sejumlah dosen dengan pistol mainan . Ia mengakui perbuatannya itu sebagai bentuk kekesalan terhadap bawahannya yang tidak melaksanakan perintahnya selaku pembantu rektor . Lantaran merasa perintahnya tidak digubris , Nataniel kemudian marah dan mencabut pistol korek api yang biasa dibawanya serta mengancam akan menembak para bawahannya . Karena tak mengetahui benda yang diacungkan itu pistol korek api , para karyawan ketakutan . Sebagian dari mereka kemudian melapor ke polisi karena merasa jiwanya terancam . Berbekal laporan itu , polisi lalu membawa Nataniel ke Markas Kepolisian Resor Pulau Ambon dan Pulau-pulau Lease untuk diperiksa . ( ANS/Juhri Samanery ) .',
  'clean_summary': 'Lantaran tak digubris perintahnya , Purek II Sekolah T

In [12]:
!pip install Sastrawi
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

import re

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 4.5 MB/s eta 0:00:00


In [16]:
# function lower
def to_lowercase(text):
    return text.lower()

# funtion stopword
factory = StopWordRemoverFactory()
stopword_remover = factory.create_stop_word_remover()

def remove_stopwords(text):
    return stopword_remover.remove(text)


def fix_punctuation(text):
    # hapus spasi sebelum tanda baca
    text = re.sub(r'\s+([.,:;!?])', r'\1', text)

    # rapihin spasi berlebih
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

def remove_header(text):
    # hapus dari "liputan..." sampai ":" pertama di awal kalimat
    text = re.sub(r'^liputan\S*.*?:\s*', '', text, flags=re.IGNORECASE)
    return text

def remove_baca(text):
    text = re.sub(r'\[\s*baca.*?\]', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def clean_parentheses(text):

    pattern = r'\([^)]*liputan\s?6[^)]*\)'

    cleaned_text = re.sub(pattern, '', text, flags=re.IGNORECASE)

    # rapihkan spasi
    cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()

    return cleaned_text

def multi_titik(text):
    # 1. Ubah titik dobel (.. atau . . atau .   .) jadi satu titik
    text = re.sub(r'\.\s*\.+', '.', text)

    # 2. Rapihkan spasi
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Penggabungan
def clean_text(text):
    text = text.lower()
    text = fix_punctuation(text)
    text = stopword_remover.remove(text)
    text = remove_header(text)
    text = remove_baca(text)
    text = clean_parentheses(text)
    text = multi_titik(text)
    return text

In [17]:
#Apply cleaning

def apply_cleaning(data):
    new_data = []

    for item in data:
        article = clean_text(item["clean_article"])
        summary = clean_text(item["clean_summary"])

        new_data.append({
            "article": article,
            "summary": summary,
            "extractive_summary": item["extractive_summary"]
        })

    return new_data

train_text_clean = apply_cleaning(train_data_convert)
dev_text_clean = apply_cleaning(dev_data_convert)
test_text_clean = apply_cleaning(test_data_convert)

In [18]:
idx = 9  # ambil data index

print("=== ORIGINAL (convert) ===")
print(train_data_convert[idx]["clean_article"])

print("\n=== CLEANED ===")
print(train_text_clean[idx]["article"])

=== ORIGINAL (convert) ===
Liputan6 . com , Jakarta : Kondisi Letnan Kolonel Ananta , anggota Marinir yang tertembak saat menangkap Syam Ahmad Sanusi , pembunuh Direktur Utama PT Asaba di Pandeglang , Banten sudah mulai pulih , Sabtu ( 18/8 ) . Ananta tertembak di dada kiri hingga punggung saat hendak menangkap Syam kemarin . Dua rekannya juga ditembak oleh Syam dan kini masih dirawat di ruang instalasi gawat darurat Rumah Sakit Al Mintoharjo , Jakarta Pusat . Dalam penyergapan itu , Syam akhirnya ditembak mati oleh tim gabungan marinir dan Kepolisian Daerah Metro Jaya [ baca : Pelarian Pembunuh Itu Berakhir di Gubuk ] . Bekas komandan pasukan elit Marinir itu buron sejak tahun 2005 setelah bersama rekannya menembak mati Dirut PT Asaba . Dengan tewasnya Syam maka pelaku kasus Asaba sudah tertangkap semua . Gunawan kini mendekam di Nusakambangan dan Suud Rusli di Pangkalan Utama AL III Jakarta . ( IAN/Tim Liputan 6 SCTV ) .

=== CLEANED ===
kondisi letnan kolonel ananta, anggota marinir

In [19]:
from datasets import Dataset

In [20]:
# Convert ke HuggingFace Dataset

train_text_dataset = Dataset.from_list(train_data_convert)
dev_text_dataset   = Dataset.from_list(dev_data_convert)
test_text_dataset  = Dataset.from_list(test_data_convert)

train_clean_dataset = Dataset.from_list(train_text_clean)
dev_clean_dataset   = Dataset.from_list(dev_text_clean)
test_clean_dataset  = Dataset.from_list(test_text_clean)

In [ ]:
print(train_clean_dataset)
print(train_clean_dataset[0:5])

In [ ]:
for i in range(3):
    print(train_text_dataset[i])
    print("---"*50)
    print(train_clean_dataset[i])
    print("===="*50)

In [ ]:
# Simpan Ke JSON untuk keperluan upload kaggle

with open("/content/drive/MyDrive/Colab Notebooks/Data/Project02/train_data_convert.json", "w") as f:
    json.dump(train_data_convert, f)

with open("/content/drive/MyDrive/Colab Notebooks/Data/Project02/dev_data_convert.json", "w") as f:
    json.dump(dev_data_convert, f)

with open("/content/drive/MyDrive/Colab Notebooks/Data/Project02/test_data_convert.json", "w") as f:
    json.dump(test_data_convert, f)

with open("/content/drive/MyDrive/Colab Notebooks/Data/Project02/train_text_clean.json", "w") as f:
    json.dump(train_text_clean, f)

with open("/content/drive/MyDrive/Colab Notebooks/Data/Project02/dev_text_clean.json", "w") as f:
    json.dump(dev_text_clean, f)

with open("/content/drive/MyDrive/Colab Notebooks/Data/Project02/test_text_clean.json", "w") as f:
    json.dump(test_text_clean, f)